# JSON for Beginners

JSON is one of the most common formats you will meet when working with APIs, config files, and data pipelines. In this notebook, we will build a simple mental model first, then practice the tools you will use most often in Python.

By the end of this notebook, you should be able to:
- explain what JSON is and why APIs use it,
- convert Python objects to and from JSON strings,
- save JSON to a file and load it back again,
- inspect and update parsed JSON data,
- and validate JSON with Pydantic models.


> **What you should already know:** You should be comfortable with basic Python variables, lists, and dictionaries.


## Visual Overview

```mermaid
graph TD
    JSON[JSON text]
    PY[Python objects]
    FILE[JSON file]
    API[API response]
    MODEL[Pydantic model]

    JSON -->|json.loads| PY
    FILE -->|json.load| PY
    PY -->|json.dump| FILE
    PY -->|json.dumps| JSON
    API -->|response.json| PY
    PY -->|validate| MODEL
```


In [2]:
import json
from pathlib import Path

import requests
from pydantic import BaseModel, ValidationError

## JSON Mental Model

JSON stands for **JavaScript Object Notation**. It is a text format for storing and exchanging structured data, which is why it appears so often in APIs.

A good first mental model is this: **JSON is text**, while **Python dictionaries and lists are Python objects in memory**.

Here is a small JSON example:

```json
{
  "name": "Ada Lovelace",
  "age": 36,
  "skills": ["math", "writing"],
  "active": true,
  "manager": null
}
```

When Python reads JSON, it maps values like this:

| JSON | Python |
| --- | --- |
| object | `dict` |
| array | `list` |
| string | `str` |
| number | `int` or `float` |
| true / false | `True` / `False` |
| null | `None` |

Notice how the JSON example uses lowercase `true` and `null`. In Python code, the matching values are `True` and `None`.


## Work With JSON in Memory First

Why start here? Because `dumps()` and `loads()` let you understand the conversion itself before you add files into the picture.

- `json.dumps(...)` turns a Python object into a JSON **string**
- `json.loads(...)` turns a JSON **string** into a Python object


In [3]:
profile = {
    "name": "Ada Lovelace",
    "age": 36,
    "skills": ["math", "writing"],
    "active": True,
    "manager": None,
}

profile_json = json.dumps(profile, indent=2)
profile_roundtrip = json.loads(profile_json)

print("Python object type:", type(profile).__name__)
print("JSON string type:", type(profile_json).__name__)
print("Round-trip type:", type(profile_roundtrip).__name__)
print()
print(profile_json)

Python object type: dict
JSON string type: str
Round-trip type: dict

{
  "name": "Ada Lovelace",
  "age": 36,
  "skills": [
    "math",
    "writing"
  ],
  "active": true,
  "manager": null
}


**Expected result:** you should see a nicely formatted JSON string. The original value is a Python `dict`, but `profile_json` is a plain string that happens to contain JSON text.


In [4]:
blackjack_hand = (8, "Q")
encoded_hand = json.dumps(blackjack_hand)
decoded_hand = json.loads(encoded_hand)

print("Original value:", blackjack_hand, type(blackjack_hand).__name__)
print("After dumps():", encoded_hand, type(encoded_hand).__name__)
print("After loads():", decoded_hand, type(decoded_hand).__name__)
print("Same value without type conversion?", blackjack_hand == decoded_hand)
print(
    "Same value after converting back to tuple?", blackjack_hand == tuple(decoded_hand)
)

Original value: (8, 'Q') tuple
After dumps(): [8, "Q"] str
After loads(): [8, 'Q'] list
Same value without type conversion? False
Same value after converting back to tuple? True


> **Checkpoint:**
> Before moving on, make sure this distinction feels clear:
> - `dumps()` and `loads()` work with **strings in memory**
> - after a JSON round trip, some Python types can come back slightly differently, such as a tuple becoming a list


## Save JSON to a File and Load It Back

Now that the string-based conversion makes sense, we can add file I/O.

- `json.dump(...)` writes JSON to a file
- `json.load(...)` reads JSON from a file


In [7]:
Path("data").mkdir(exist_ok=True)

persons_seed = [
    {
        "name": "John Doe",
        "age": 30,
        "email": "john.doe@example.com",
        "address": {
            "street": "123 Main St",
            "city": "Anytown",
            "state": "CA",
            "zip": "12345",
        },
        "phoneNumbers": [
            {"type": "home", "number": "555-555-1234"},
            {"type": "work", "number": "555-555-5678"},
        ],
        "isEmployee": True,
        "salary": None,
    }
]

with open("data/example.json", "w") as f:
    json.dump(persons_seed, f, indent=2)

with open("data/example.json") as f:
    persons = json.load(f)

print("File written to:", Path("data/example.json").resolve())
print("Loaded object type:", type(persons).__name__)
print("First record:")
print(json.dumps(persons[0], indent=2))

File written to: C:\Users\thheg\bootcamps\week17\mle-api-docker\data\example.json
Loaded object type: list
First record:
{
  "name": "John Doe",
  "age": 30,
  "email": "john.doe@example.com",
  "address": {
    "street": "123 Main St",
    "city": "Anytown",
    "state": "CA",
    "zip": "12345"
  },
  "phoneNumbers": [
    {
      "type": "home",
      "number": "555-555-1234"
    },
    {
      "type": "work",
      "number": "555-555-5678"
    }
  ],
  "isEmployee": true,
  "salary": null
}


**Expected result:** the notebook should create `data/example.json`, then load it back as a Python list of dictionaries.


## Read and Modify Parsed JSON

Once JSON has been loaded into Python, you work with it using normal Python list and dictionary operations.


In [8]:
print("Name:", persons[0].get("name"))
print("First phone number:", persons[0]["phoneNumbers"][0]["number"])

persons[0].update({"name": "Jane Doe"})

new_person = {
    "name": "Sam Rivera",
    "age": 25,
    "email": "sam.rivera@example.com",
    "address": {
        "street": "456 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": "12345",
    },
    "phoneNumbers": [{"type": "mobile", "number": "555-555-4321"}],
    "isEmployee": False,
    "salary": None,
}

persons.append(new_person)

print()
print("Updated first name:", persons[0]["name"])
print("Number of people:", len(persons))
print("Second person name:", persons[1]["name"])

Name: John Doe
First phone number: 555-555-1234

Updated first name: Jane Doe
Number of people: 2
Second person name: Sam Rivera


**Expected result:** you should see values pulled from nested JSON data, an updated name, and a second person added to the list.

> **Checkpoint:**
> If `persons` came from `json.load(...)`, why can you use list indexing and dictionary methods on it?  
> **Answer:** Because after parsing, JSON data becomes normal Python objects.


## A Real-World API Example

Most APIs send JSON in their responses. The `requests` library makes it easy to fetch that data.

In practice, `response.json()` is the normal, convenient way to parse a JSON response. `json.loads(response.text)` is the more explicit lower-level path when you want to start from the raw text yourself.


In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/todos", timeout=10)
response.raise_for_status() #error handling, throws HTTPError exception

todos_from_response = response.json()
todos_from_text = json.loads(response.text)

print("Status code:", response.status_code)
print("Objects match:", todos_from_response == todos_from_text)
print("Total todos:", len(todos_from_response))
print("First todo:")
print(json.dumps(todos_from_response[0], indent=2))

Status code: 200
Objects match: True
Total todos: 200
First todo:
{
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}


**Expected result:** both parsing approaches should produce the same Python data. In everyday code, prefer `response.json()` unless you specifically need the raw response text first.


## Validate JSON With Pydantic

Parsing tells you how to turn JSON into Python objects. Validation answers a different question: **does this data match the shape I expect?**

This matters in APIs because incoming or external data can be missing fields, use the wrong types, or contain values you were not expecting.


In [10]:
class Address(BaseModel):
    street: str
    city: str
    state: str
    zip: str


class User(BaseModel):
    name: str
    age: int
    email: str
    address: Address
    phone: str


valid_user_json = """{
    "name": "John Doe",
    "age": 30,
    "email": "john.doe@example.com",
    "address": {
        "street": "123 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": "12345"
    },
    "phone": "555-555-1234"
}"""

invalid_user_json = """{
    "name": "Broken Example",
    "age": "thirty",
    "email": "broken@example.com",
    "address": {
        "street": "123 Main St",
        "city": "Anytown",
        "state": "CA",
        "zip": 12345
    }
}"""

In [11]:
valid_user = User.model_validate_json(valid_user_json)
print("Validated user:")
print(valid_user)
print()

valid_user_dict = json.loads(valid_user_json)
validated_from_dict = User.model_validate(valid_user_dict)
print("Validated from Python dict:")
print(validated_from_dict)
print()

print("Invalid example errors:")
try:
    User.model_validate_json(invalid_user_json)
except ValidationError as e:
    print(e)

Validated user:
name='John Doe' age=30 email='john.doe@example.com' address=Address(street='123 Main St', city='Anytown', state='CA', zip='12345') phone='555-555-1234'

Validated from Python dict:
name='John Doe' age=30 email='john.doe@example.com' address=Address(street='123 Main St', city='Anytown', state='CA', zip='12345') phone='555-555-1234'

Invalid example errors:
3 validation errors for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/int_parsing
address.zip
  Input should be a valid string [type=string_type, input_value=12345, input_type=int]
    For further information visit https://errors.pydantic.dev/2.8/v/string_type
phone
  Field required [type=missing, input_value={'name': 'Broken Example'...e': 'CA', 'zip': 12345}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing


**Expected result:** the valid example should parse into a `User` object, while the invalid example should show readable validation errors.

> **Checkpoint:**
> At this point, try saying the difference out loud:
> - parsing converts data between formats
> - validation checks whether the parsed data fits the schema you expect


## Recap

Here are the big distinctions to keep straight:

- **JSON vs Python objects:** JSON is text; Python dictionaries and lists live in memory.
- **`dump/load` vs `dumps/loads`:** the versions without `s` work with files, and the versions with `s` work with strings.
- **`response.json()` vs `json.loads(response.text)`:** both can parse API JSON, but `response.json()` is usually the simpler choice.
- **Parsing vs validation:** parsing converts data, validation checks whether the data matches your expected structure.


## Capstone Practice

Try this workflow end to end:
1. Fetch todos from JSONPlaceholder.
2. Inspect one item.
3. Define a Pydantic model for a todo.
4. Validate a small subset of the response.
5. Save the validated todos to a JSON file.

Read the next code cell like a reference solution and compare it with how you would approach the task yourself.


In [15]:
class Todo(BaseModel):
    userId: int
    id: int
    title: str
    completed: bool


sample_todo = todos_from_response[0]
validated_todos = [Todo.model_validate(item) for item in todos_from_response[:5]]
validated_todo_dicts = [todo.model_dump() for todo in validated_todos]
print(validated_todos)
print(validated_todo_dicts)


[Todo(userId=1, id=1, title='delectus aut autem', completed=False), Todo(userId=1, id=2, title='quis ut nam facilis et officia qui', completed=False), Todo(userId=1, id=3, title='fugiat veniam minus', completed=False), Todo(userId=1, id=4, title='et porro tempora', completed=True), Todo(userId=1, id=5, title='laboriosam mollitia et enim quasi adipisci quia provident illum', completed=False)]
[{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}, {'userId': 1, 'id': 2, 'title': 'quis ut nam facilis et officia qui', 'completed': False}, {'userId': 1, 'id': 3, 'title': 'fugiat veniam minus', 'completed': False}, {'userId': 1, 'id': 4, 'title': 'et porro tempora', 'completed': True}, {'userId': 1, 'id': 5, 'title': 'laboriosam mollitia et enim quasi adipisci quia provident illum', 'completed': False}]


In [16]:
with open("data/validated_todos.json", "w") as f:
    json.dump(validated_todo_dicts, f, indent=2)

print("Sample todo:")
print(json.dumps(sample_todo, indent=2))
print()
print("Validated", len(validated_todos), "todos")
print("Saved file:", Path("data/validated_todos.json").resolve())

Sample todo:
{
  "userId": 1,
  "id": 1,
  "title": "delectus aut autem",
  "completed": false
}

Validated 5 todos
Saved file: C:\Users\thheg\bootcamps\week17\mle-api-docker\data\validated_todos.json


**Expected result:** you should validate the first five todos successfully and create `data/validated_todos.json`.


### **Quiz**

1. **What is the main difference between JSON and a Python dictionary?**  
   [ ] JSON can only store numbers    
   [ ] Dictionaries can be sent over APIs, but JSON cannot  
   [ ] JSON is text, while a dictionary is a Python object  
   [ ] There is no difference  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** JSON is text, while a dictionary is a Python object  

     **Description:** JSON is a text-based data format used for data exchange, while a Python dictionary is an in-memory data structure. JSON must be parsed into a dictionary to be used in Python.
   </details>

---

2. **Which function turns a Python object into a JSON string?**  
   [ ] `json.load()`  
   [ ] `json.loads()`  
   [ ] `json.dump()`  
   [ ] `json.dumps()`  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** `json.dumps()`  

     **Description:** `json.dumps()` converts a Python object into a JSON-formatted string, while `json.loads()` parses a JSON string and converts it into a Python object.
   </details>

---

3. **Which pair of functions works with files instead of strings?**  
   [ ] `dump()` and `load()`  
   [ ] `dumps()` and `loads()`  
   [ ] `print()` and `input()`  
   [ ] `read()` and `write()`  

   <details>
     <summary>Show Answer</summary>
     
     **Correct Answer:** `dump()` and `load()`  

     **Description:** `dump()` and `load()` are used to write JSON data to a file and read JSON data from a file, respectively, while `dumps()` and `loads()` work with JSON as strings.
   </details>

---

4. **Why might a tuple come back as a list after a JSON round trip?**  
   [ ] JSON does not have a tuple type  
   [ ] Python automatically converts all lists into tuples  
   [ ] `json.dumps()` removes brackets  
   [ ] JSON only supports strings  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** JSON does not have a tuple type  

     **Description:** JSON only supports arrays, which map to Python lists, so Python tuples are decoded back as lists.
   </details>

---

5. **When reading JSON from an API response, what is usually the simplest option?**  
   [ ] `json.loads(response.text)`  
   [ ] `json.dump(response)`  
   [ ] `response.json()`  
   [ ] `response.text()`  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** `response.json()`  

     **Description:** In practice, `response.json()` is the convenient way to parse JSON from an HTTP response. While `json.loads(response.text)` achieves the same result starting from the raw text, `response.json()` is simpler and preferred in everyday code.
   </details>

---

6. **What does Pydantic add on top of basic JSON parsing?**  
   [ ] It compresses JSON files  
   [ ] It replaces all dictionaries with tuples  
   [ ] It removes the need for Python types  
   [ ] It validates data against a model  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** It validates data against a model  

     **Description:** Pydantic validates that incoming data matches the structure and types defined in your model.
   </details>
